In [4]:
pip install requests beautifulsoup4 pandas openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re
from collections import OrderedDict

# =========================
# CONFIG
# =========================

BASE_URL = "https://job.kaspi.kz"
SEARCH_TEMPLATE = "https://job.kaspi.kz/search?categories=33__34&page={}"
MAX_PAGES = 50
OUTPUT_FILE = "kaspi_it_market_analysis.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8",
    "Referer": "https://job.kaspi.kz/"
}

# =========================
# UTILS
# =========================

def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def fetch(session: requests.Session, url: str) -> str | None:
    """Безопасный HTTP-запрос"""
    try:
        response = session.get(url, timeout=10)
        if response.status_code != 200:
            print(f"[WARN] {response.status_code} -> {url}")
            return None
        return response.text
    except requests.RequestException as e:
        print(f"[ERROR] Request failed: {e}")
        return None


def polite_sleep(a=0.6, b=1.4):
    time.sleep(random.uniform(a, b))

# =========================
# SCRAPING
# =========================

def get_all_vacancy_links(session: requests.Session) -> list[str]:
    """Собирает ссылки на вакансии со всех страниц"""
    links = []

    for page in range(1, MAX_PAGES + 1):
        print(f"Сканирую страницу поиска: {page}")
        html = fetch(session, SEARCH_TEMPLATE.format(page))
        if not html:
            break

        soup = BeautifulSoup(html, "html.parser")
        cards = soup.find_all("div", class_="vacancy")

        if not cards:
            print("Вакансии закончились.")
            break

        for card in cards:
            a = card.find("a", href=True)
            if a:
                links.append(BASE_URL + a["href"])

        polite_sleep()

    # Убираем дубликаты, сохраняя порядок
    return list(OrderedDict.fromkeys(links))


def parse_vacancy_details(session: requests.Session, url: str) -> dict | None:
    """Парсит детальную страницу вакансии"""
    html = fetch(session, url)
    if not html:
        return None

    soup = BeautifulSoup(html, "html.parser")

    # Заголовок
    title = soup.find("h1", class_="vacancy-single__head__content__title")
    salary = soup.find("div", class_="vacancy-single__head__content__salary__label")
    city = soup.find("div", class_="vacancy-single__head__content__salary__cities")

    experience = ""
    schedule = ""

    filters = soup.find_all(
        "div",
        class_="vacancy-single__head__content__filter-items__label"
    )

    for f in filters:
        text = clean_text(f.get_text()).lower()

        # Опыт (есть числа и слова "год/лет")
        if re.search(r"\d", text) and any(w in text for w in ["год", "лет"]):
            experience = text

        # График
        elif any(w in text for w in [
            "график", "удал", "офис", "гибк", "полный", "частич"
        ]):
            schedule = text
        
    # Описание и навыки
    desc_container = soup.find("div", class_="vacancy-single__body__desc")

    full_description = ""
    skills = []

    if desc_container:
        full_description = clean_text(desc_container.get_text(" "))

        for item in desc_container.find_all(["li", "p"]):
            t = clean_text(item.get_text())
            if t.startswith(("•", "-", "—")) or item.name == "li":
                skill = t.lstrip("•-— ").strip()
                if skill and len(skill) < 120:
                    skills.append(skill)

    return {
        "Title": clean_text(title.get_text()) if title else "N/A",
        "Salary_raw": clean_text(salary.get_text()) if salary else "Не указана",
        "City": clean_text(city.get_text()) if city else "N/A",
        "Experience": experience,
        "Schedule": schedule,
        "Skills": " | ".join(skills),
        "Full_description": full_description,
        "URL": url
    }

# =========================
# MAIN PIPELINE
# =========================

def main():
    session = requests.Session()
    session.headers.update(HEADERS)

    links = get_all_vacancy_links(session)
    print(f"\nНайдено вакансий: {len(links)}\n")

    results = []

    for i, link in enumerate(links, start=1):
        print(f"[{i}/{len(links)}] Парсинг: {link}")
        data = parse_vacancy_details(session, link)
        if data:
            results.append(data)
        polite_sleep(0.4, 1.0)

    df = pd.DataFrame(results)
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\nГотово. Данные сохранены в '{OUTPUT_FILE}'")

if __name__ == "__main__":
    main()

Сканирую страницу поиска: 1
Сканирую страницу поиска: 2
Сканирую страницу поиска: 3
Сканирую страницу поиска: 4
Вакансии закончились.

Найдено вакансий: 53

[1/53] Парсинг: https://job.kaspi.kz/vacancy/devops-engineer-kaspi-shop
[2/53] Парсинг: https://job.kaspi.kz/vacancy/middle-java-developer-7j
[3/53] Парсинг: https://job.kaspi.kz/vacancy/servis-inzhener
[4/53] Парсинг: https://job.kaspi.kz/vacancy/ekspert-service-desk
[5/53] Парсинг: https://job.kaspi.kz/vacancy/tehnicheskiy-pisatel
[6/53] Парсинг: https://job.kaspi.kz/vacancy/java-developer-messenger
[7/53] Парсинг: https://job.kaspi.kz/vacancy/middlesenior-1c-developer
[8/53] Парсинг: https://job.kaspi.kz/vacancy/cnet-developer
[9/53] Парсинг: https://job.kaspi.kz/vacancy/middle-data-engineer
[10/53] Парсинг: https://job.kaspi.kz/vacancy/product-analyst-le
[11/53] Парсинг: https://job.kaspi.kz/vacancy/application-security-engineer-middle-senior
[12/53] Парсинг: https://job.kaspi.kz/vacancy/ios-developer-zt
[13/53] Парсинг: https:

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import random

# --- КОНФИГУРАЦИЯ ---
BASE_URL = "https://job.kaspi.kz"
# Категории 33 (IT) и 34 (Data)
SEARCH_URL_TEMPLATE = "https://job.kaspi.kz/search?categories=33__34&page={}"

# Реалистичные заголовки браузера
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://job.kaspi.kz/',
    'Connection': 'keep-alive'
}

# --- ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---

def clean_text(text):
    """Очищает текст от мусора, лишних пробелов и артефактов верстки"""
    if not text:
        return ""
    text = text.replace('\xa0', ' ') # Убираем неразрывные пробелы
    text = re.sub(r'\s+', ' ', text) # Схлопываем множественные пробелы и переносы
    return text.strip()

def get_vacancy_links(session):
    """Собирает все уникальные ссылки на вакансии со всех страниц пагинации"""
    all_links = []
    page = 1
    
    while True:
        url = SEARCH_URL_TEMPLATE.format(page)
        print(f"[ПОИСК] Сканирую страницу {page}...")
        
        try:
            response = session.get(url, timeout=15)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Находим все карточки вакансий
            cards = soup.find_all('div', class_='vacancy')
            
            if not cards:
                print(f"[ИНФО] На странице {page} вакансий нет. Завершаю сбор ссылок.")
                break
            
            for card in cards:
                link_tag = card.find('a', href=True)
                if link_tag:
                    full_link = BASE_URL + link_tag['href']
                    all_links.append(full_link)
            
            page += 1
            time.sleep(random.uniform(1, 2)) # Вежливая пауза
            
        except Exception as e:
            print(f"[ОШИБКА] Не удалось прочитать страницу {page}: {e}")
            break
            
    return list(dict.fromkeys(all_links)) # Удаляем дубликаты, сохраняя порядок

def parse_vacancy_page(session, url):
    """Парсит детальную информацию внутри конкретной вакансии"""
    try:
        response = session.get(url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 1. Заголовок и база
        title = soup.find('h1', class_='vacancy-single__head__content__title')
        salary = soup.find('div', class_='vacancy-single__head__content__salary__label')
        city = soup.find('div', class_='vacancy-single__head__content__salary__cities')
        
        # 2. Опыт и график (фильтры под заголовком)
        filter_tags = soup.find_all('div', class_='vacancy-single__head__content__filter-items__label')
        schedule = filter_tags[0].get_text(strip=True) if len(filter_tags) > 0 else "N/A"
        experience = filter_tags[1].get_text(strip=True) if len(filter_tags) > 1 else "N/A"
        
        # 3. Детальное описание и навыки
        desc_block = soup.find('div', class_='vacancy-single__body__desc')
        full_description = ""
        skills_extracted = []
        
        if desc_block:
            full_description = clean_text(desc_block.get_text(" "))
            
            # Собираем все требования из списков и параграфов с буллитами
            # Это самое важное для анализа стека (Python, SQL, Docker и т.д.)
            items = desc_block.find_all(['li', 'p'])
            for item in items:
                text = item.get_text(strip=True)
                if item.name == 'li' or text.startswith('•') or text.startswith('-'):
                    clean_item = text.lstrip('•- ').strip()
                    if clean_item and len(clean_item) > 2:
                        skills_extracted.append(clean_item)

        return {
            'Position': clean_text(title.get_text()) if title else "N/A",
            'Salary': clean_text(salary.get_text()) if salary else "Не указана",
            'City': clean_text(city.get_text()) if city else "N/A",
            'Experience': experience,
            'Schedule': schedule,
            'Skills_List': " | ".join(skills_extracted),
            'Full_Description': full_description,
            'URL': url
        }
        
    except Exception as e:
        print(f"[ОШИБКА] Не удалось распарсить {url}: {e}")
        return None

# --- ГЛАВНЫЙ ЦИКЛ ---

def main():
    start_time = time.time()
    
    # Используем Session для повторного использования TCP-соединения (быстрее и надежнее)
    with requests.Session() as session:
        session.headers.update(HEADERS)
        
        # Шаг 1: Сбор всех ссылок
        links = get_all_vacancy_links(session)
        print(f"\n[ИТОГ] Найдено вакансий для анализа: {len(links)}\n")
        
        # Шаг 2: Сбор данных по каждой ссылке
        all_data = []
        for i, link in enumerate(links, 1):
            print(f"[{i}/{len(links)}] Обработка: {link.split('/')[-1]}")
            
            vacancy_info = parse_vacancy_page(session, link)
            if vacancy_info:
                all_data.append(vacancy_info)
            
            # Рандомная пауза, чтобы не триггерить защиту
            time.sleep(random.uniform(0.5, 1.2))
            
        # Шаг 3: Сохранение результатов
        if all_data:
            df = pd.DataFrame(all_data)
            
            # Сохраняем в CSV с кодировкой utf-8-sig (чтобы Excel сразу видел кириллицу)
            output_file = "kaspi_it_vacancies_2024.csv"
            df.to_csv(output_file, index=False, encoding='utf-8-sig')
            
            
            end_time = time.time()
            duration = round(end_time - start_time, 1)
            print(f"\n{'='*30}")
            print(f"ГОТОВО! Собрано {len(all_data)} вакансий за {duration} сек.")
            print(f"Файл сохранен: {output_file}")
            print(f"{'='*30}")
        else:
            print("Данные не были собраны.")

if __name__ == "__main__":
    main()

Сканирую страницу поиска: 1
Сканирую страницу поиска: 2
Сканирую страницу поиска: 3
Сканирую страницу поиска: 4
Вакансии закончились.

[ИТОГ] Найдено вакансий для анализа: 53

[1/53] Обработка: devops-engineer-kaspi-shop
[2/53] Обработка: middle-java-developer-7j
[3/53] Обработка: servis-inzhener
[4/53] Обработка: ekspert-service-desk
[5/53] Обработка: tehnicheskiy-pisatel
[6/53] Обработка: java-developer-messenger
[7/53] Обработка: middlesenior-1c-developer
[8/53] Обработка: cnet-developer
[9/53] Обработка: middle-data-engineer
[10/53] Обработка: product-analyst-le
[11/53] Обработка: application-security-engineer-middle-senior
[12/53] Обработка: ios-developer-zt
[13/53] Обработка: data-analyst-3i
[14/53] Обработка: product-manager-rekomendatelnyh-sistem
[15/53] Обработка: senior-design-system-designer
[16/53] Обработка: android-developer
[17/53] Обработка: net-gosuslugi-kaspi
[18/53] Обработка: net-developer-kaspi-marketing
[19/53] Обработка: net-developer-pg
[20/53] Обработка: eksper